# Protein Subcellular Localization with ESM-2

Fine-tune Meta AI's **ESM-2** protein language model to predict where a protein
localizes in the cell (nucleus, mitochondrion, membrane, ...) directly from its
amino-acid sequence.

**Dataset:** [`proteinea/deeploc`](https://huggingface.co/datasets/proteinea/deeploc) —
the DeepLoc benchmark (Almagro Armenteros et al., 2017): ~8.5k SwissProt proteins with
experimentally-supported localization across 10 compartments.

**Runtime:** free Colab **T4** GPU (`Runtime -> Change runtime type -> T4 GPU`).
Runs end-to-end in roughly 15-30 min with the 150M model.

## 1 · Setup

In [ ]:
!pip -q install -U transformers datasets umap-learn scikit-learn seaborn accelerate

In [ ]:
import torch, numpy as np, pandas as pd
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU only')

## 2 · Load the DeepLoc benchmark

The upstream dataset ships `train`/`test`; we carve a stratified validation split so
model selection never touches the test set.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

raw = load_dataset('proteinea/deeploc')
train_full = raw['train'].to_pandas()[['input','loc']].rename(columns={'input':'sequence','loc':'label'})
test = raw['test'].to_pandas()[['input','loc']].rename(columns={'input':'sequence','loc':'label'})
train_full = train_full.dropna().drop_duplicates('sequence')
test = test.dropna().drop_duplicates('sequence')
train, val = train_test_split(train_full, test_size=0.1, stratify=train_full['label'], random_state=42)
print(f'{len(train)} train / {len(val)} val / {len(test)} test')
train['label'].value_counts()

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style='whitegrid')
counts = train['label'].value_counts()
plt.figure(figsize=(9,4)); sns.barplot(x=counts.values, y=counts.index, palette='viridis')
plt.title('Class distribution (train) — note the imbalance'); plt.xlabel('count'); plt.tight_layout(); plt.show()

## 3 · Tokenize with the ESM-2 tokenizer

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'facebook/esm2_t30_150M_UR50D'  # T4-friendly. Try esm2_t33_650M_UR50D for best accuracy.
MAX_LENGTH = 512

labels = sorted(train['label'].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
num_labels = len(labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    def tok(b):
        t = tokenizer(b['sequence'], truncation=True, max_length=MAX_LENGTH)
        t['labels'] = [label2id[l] for l in b['label']]
        return t
    return ds.map(tok, batched=True, remove_columns=['sequence','label'])

train_ds, val_ds, test_ds = to_ds(train), to_ds(val), to_ds(test)

## 4 · Fine-tune ESM-2

We add a class-weighted cross-entropy loss (inverse-frequency) so rare compartments
like Peroxisome aren't ignored, and select the checkpoint with the best **macro-F1**.

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, label2id=label2id, id2label=id2label)

counts = train['label'].map(label2id).value_counts().reindex(range(num_labels), fill_value=0).sort_index().values
w = counts.sum() / (num_labels * np.clip(counts, 1, None)); w = w / w.mean()
class_weights = torch.tensor(w, dtype=torch.float32)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'macro_f1': f1_score(p.label_ids, preds, average='macro'),
            'mcc': matthews_corrcoef(p.label_ids, preds)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

args = TrainingArguments(
    output_dir='outputs', num_train_epochs=3,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    gradient_accumulation_steps=2, learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=1,
    fp16=(device=='cuda'), logging_steps=20, report_to='none')

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()

## 5 · Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_metrics = trainer.evaluate(test_ds)
print({k: round(v,4) for k,v in test_metrics.items() if isinstance(v,float)})

pred = trainer.predict(test_ds)
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(9,8))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, values_format='.2f', cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix (row-normalized) — Test'); plt.tight_layout(); plt.show()

## 6 · Why it works: the embedding map

Mean-pool ESM-2's final hidden states into one vector per protein and project to 2-D with
UMAP. Proteins cluster by compartment — the localization signal is already latent in the
language-model representation, which is exactly why fine-tuning transfers so well.

In [ ]:
from transformers import AutoModel
enc = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

@torch.no_grad()
def embed(seqs, bs=16):
    vecs=[]
    for i in range(0,len(seqs),bs):
        b=tokenizer(list(seqs[i:i+bs]), return_tensors='pt', truncation=True, padding=True, max_length=MAX_LENGTH).to(device)
        h=enc(**b).last_hidden_state; m=b['attention_mask'].unsqueeze(-1).float()
        vecs.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).float().cpu().numpy())
    return np.concatenate(vecs)

emb = embed(test['sequence'].tolist())
print('embeddings:', emb.shape)

In [ ]:
import umap
xy = umap.UMAP(n_neighbors=25, min_dist=0.3, metric='cosine', random_state=42).fit_transform(emb)
yt = test['label'].values
plt.figure(figsize=(11,8.5))
for c,color in zip(labels, sns.color_palette('tab10', len(labels))):
    m = yt==c; plt.scatter(xy[m,0], xy[m,1], s=10, alpha=0.6, color=color, label=c, linewidths=0)
plt.legend(markerscale=2, fontsize=9); plt.title('ESM-2 embeddings (UMAP) colored by subcellular location')
plt.tight_layout(); plt.show()

## 7 · Save & try it

In [ ]:
trainer.save_model('outputs/best_model'); tokenizer.save_pretrained('outputs/best_model')

@torch.no_grad()
def classify(seq, k=3):
    m = model.eval().to(device)
    x = tokenizer(seq, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(device)
    p = torch.softmax(m(**x).logits, -1)[0]
    top = torch.topk(p, k)
    return [(id2label[i.item()], round(v.item(),4)) for v,i in zip(*top)]

# Example: human insulin (INS) — secreted/extracellular
insulin = 'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN'
classify(insulin)

## 8 · Refresh the live dashboard

Regenerate `results.js` from **this fine-tuned model** and download it. Drop the file into
`docs/` in the repo (replacing the baseline one) and push — GitHub Pages updates the live
dashboard within a minute, now showing the fine-tuned model's real numbers.

In [ ]:
import json
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

logits = pred.predictions
probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
conf = probs.max(1)

met = {'accuracy': float(accuracy_score(y_true, y_pred)),
       'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
       'weighted_f1': float(f1_score(y_true, y_pred, average='weighted')),
       'mcc': float(matthews_corrcoef(y_true, y_pred)), 'n_test': int(len(y_true))}

ids = list(range(len(labels)))
pr, rc, f1c, sup = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
per_class = [{'label': labels[i], 'precision': float(pr[i]), 'recall': float(rc[i]),
              'f1': float(f1c[i]), 'support': int(sup[i])} for i in ids]
cm = confusion_matrix(y_true, y_pred, labels=ids)
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
tr = train['label'].value_counts().to_dict()
distribution = [{'label': c, 'count': int(tr.get(c, 0))} for c in labels]

xy = umap.UMAP(n_neighbors=25, min_dist=0.3, metric='cosine', random_state=42).fit_transform(emb)
points = [{'x': round(float(xy[i,0]),3), 'y': round(float(xy[i,1]),3),
           't': int(y_true[i]), 'p': int(probs[i].argmax()), 'c': round(float(conf[i]),3)}
          for i in range(len(y_true))]

seqs = test['sequence'].tolist(); examples = []
for ci, c in enumerate(labels):
    mask = (y_true==ci) & (y_pred==ci)
    if not mask.any(): continue
    idxs = np.where(mask)[0]; best = idxs[conf[idxs].argmax()]
    order = probs[best].argsort()[::-1][:3]
    examples.append({'true': c, 'length': len(seqs[best]), 'preview': seqs[best][:60],
        'top3': [{'label': labels[j], 'prob': round(float(probs[best,j]),3)} for j in order]})

results = {'model': MODEL_NAME, 'source': 'fine-tuned ESM-2 (softmax classifier)',
  'dataset': {'name': 'DeepLoc (proteinea/deeploc)', 'n_train': int(len(train)),
              'n_test': int(len(y_true)), 'n_classes': len(labels)},
  'classes': labels, 'metrics': met, 'per_class': per_class,
  'confusion': {'counts': cm.astype(int).tolist(), 'normalized': np.round(cm_norm,3).tolist()},
  'distribution': distribution, 'points': points, 'examples': examples}

with open('results.js','w') as fh:
    fh.write('// Auto-generated by the fine-tuning notebook.\n')
    fh.write('window.RESULTS = ' + json.dumps(results, separators=(',',':')) + ';\n')
print('Test accuracy %.3f | macro-F1 %.3f  ->  results.js written' % (met['accuracy'], met['macro_f1']))

from google.colab import files
files.download('results.js')

---
Model: [ESM-2](https://huggingface.co/facebook/esm2_t30_150M_UR50D) (Lin et al., 2023) ·
Data: [DeepLoc](https://services.healthtech.dtu.dk/services/DeepLoc-2.0/) (Almagro Armenteros et al., 2017)